In [2]:
import torch
import torch.nn as nn
import os, time
import numpy as np
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import googlenet
from sklearn.model_selection import train_test_split

# FX Quantization imports
from torch.ao.quantization import get_default_qconfig, QConfigMapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx


# Settings

torch.backends.quantized.engine = 'fbgemm'
DEVICE = torch.device("cpu")  # PTQ runs/evaluates on CPU
NUM_CLASSES = 200


#  Dataset & DataLoaders

# Training transform (with augmentation, like baseline)
train_transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Evaluation transform (no augmentation)
eval_transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"
full_dataset = datasets.ImageFolder(root=data_dir, transform=eval_transform)
print(f"Total images: {len(full_dataset)}, Classes: {len(full_dataset.classes)}")

# Stratified split
targets = np.array(full_dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for class_id in np.unique(targets):
    idxs = np.where(targets == class_id)[0]
    train_ids, temp_ids = train_test_split(idxs, test_size=0.15, random_state=42, shuffle=True)
    val_ids, test_ids = train_test_split(temp_ids, test_size=1/3, random_state=42, shuffle=True)
    train_idx.extend(train_ids); val_idx.extend(val_ids); test_idx.extend(test_ids)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# Datasets with separate transforms
dataset_train = datasets.ImageFolder(root=data_dir, transform=train_transform)
dataset_eval  = datasets.ImageFolder(root=data_dir, transform=eval_transform)

train_dataset = Subset(dataset_train, train_idx)
val_dataset   = Subset(dataset_eval, val_idx)
test_dataset  = Subset(dataset_eval, test_idx)

batch_size = 128
num_workers = min(8, os.cpu_count())

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)

print("DataLoaders ready")


# Load trained FP32 GoogLeNet

state_dict = torch.load("/home/h5/sapi279d/googlenet_weights.pth", map_location="cpu")

# Fix potential "_orig_mod." prefixes
new_state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

model_fp32 = googlenet(weights=None, aux_logits=True)
model_fp32.fc = nn.Linear(model_fp32.fc.in_features, NUM_CLASSES)
model_fp32.aux1.fc2 = nn.Linear(model_fp32.aux1.fc2.in_features, NUM_CLASSES)
model_fp32.aux2.fc2 = nn.Linear(model_fp32.aux2.fc2.in_features, NUM_CLASSES)

missing, unexpected = model_fp32.load_state_dict(new_state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)
print("FP32 GoogLeNet loaded")

model_fp32.eval()


# FX Graph Mode PTQ (Static Quantization)

qconfig = get_default_qconfig("fbgemm")
qconfig_mapping = QConfigMapping().set_global(qconfig)

example_inputs = (torch.randn(1, 3, 224, 224),)
prepared = prepare_fx(model_fp32, qconfig_mapping, example_inputs)

# Calibration
def calibrate_fx(model, loader, num_batches=20):
    model.eval()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches:
                break
            model(images)

calibrate_fx(prepared, train_loader, num_batches=20)
print("Calibration done")

# Convert to INT8
model_int8 = convert_fx(prepared).to("cpu")
print("Static quantization complete")


# Evaluation

def evaluate_model(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            outputs = model(images)
            if isinstance(outputs, tuple):  # handle aux outputs if any
                outputs = outputs[0]
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

acc_fp32 = evaluate_model(model_fp32, test_loader)
acc_int8 = evaluate_model(model_int8, test_loader)

print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"INT8 Accuracy: {acc_int8:.2f}%")


#  Model size & Runtime

def get_model_size(model, filename="temp.pth"):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / 1e6
    os.remove(filename)
    return size_mb

def benchmark(model, loader, num_batches=50):
    model.eval()
    start = time.time()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches: break
            _ = model(images)
    end = time.time()
    total_images = num_batches * loader.batch_size
    total_time = end - start
    return (total_time / num_batches) * 1000, (total_time / total_images) * 1000  # ms/batch, ms/image

fp32_size = get_model_size(model_fp32)
int8_size = get_model_size(model_int8)

print(f"FP32 model size: {fp32_size:.2f} MB")
print(f"INT8 model size: {int8_size:.2f} MB")

batch_ms_fp32, img_ms_fp32 = benchmark(model_fp32, test_loader)
batch_ms_int8, img_ms_int8 = benchmark(model_int8, test_loader)

print(f"FP32 Inference: {batch_ms_fp32:.2f} ms/batch | {img_ms_fp32:.2f} ms/image")
print(f"INT8 Inference: {batch_ms_int8:.2f} ms/batch | {img_ms_int8:.2f} ms/image")


Total images: 100000, Classes: 200
Train: 85000, Val: 10000, Test: 5000
✅ DataLoaders ready


/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torchvision/models/googlenet.py:47: FutureWarning: The default weight initialization of GoogleNet will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


Missing keys: []
Unexpected keys: []
✅ FP32 GoogLeNet loaded


/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torch/ao/quantization/observer.py:220: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


Calibration done ✅
Static quantization complete ✅
FP32 Accuracy: 45.10%
INT8 Accuracy: 45.02%
FP32 model size: 42.34 MB
INT8 model size: 6.03 MB
FP32 Inference: 4455.75 ms/batch | 34.81 ms/image
INT8 Inference: 2654.05 ms/batch | 20.73 ms/image
